In [ ]:
!pip install numpy pandas scikit-learn matplotlib psutil -q

In [ ]:
import os
os.makedirs('scripts', exist_ok=True)
os.makedirs('results', exist_ok=True)

In [ ]:
%%writefile scripts/__init__.py

In [ ]:
%%writefile scripts/system_info.py
import platform, socket, psutil, sys

def get_system_info() -> dict:
    uname = platform.uname()
    return {
        'hostname':    'GoogleColab',
        'os':          f'{uname.system} {uname.release}',
        'cpu':         uname.processor or platform.processor(),
        'cpu_cores':   psutil.cpu_count(logical=False) or 1,
        'cpu_threads': psutil.cpu_count(logical=True)  or 1,
        'ram_gb':      round(psutil.virtual_memory().total / (1024**3), 2),
        'python':      sys.version.split()[0],
        'architecture': platform.machine(),
    }

In [ ]:
%%writefile scripts/bench_cpu.py
import time, numpy as np, multiprocessing
from concurrent.futures import ThreadPoolExecutor

N_SMALL = 5_000_000
N_LARGE = 50_000_000
N_REPEAT = 3

def _time_it(func, *args):
    times = []
    for _ in range(N_REPEAT):
        t0 = time.perf_counter()
        func(*args)
        times.append(time.perf_counter() - t0)
    return min(times)

def _integer_ops():
    a = np.arange(N_LARGE, dtype=np.int64)
    b = np.arange(N_LARGE, dtype=np.int64) + 1
    c = a * b + a % 1000
    return c.sum()

def _float_ops():
    a = np.random.rand(N_LARGE).astype(np.float64)
    b = np.random.rand(N_LARGE).astype(np.float64)
    c = np.sqrt(a * b + a)
    return c.sum()

def _worker_task(_):
    a = np.random.rand(N_SMALL).astype(np.float64)
    return np.dot(a, a)

def _multithread_ops():
    n = multiprocessing.cpu_count()
    with ThreadPoolExecutor(max_workers=n) as ex:
        list(ex.map(_worker_task, range(n)))

def run_cpu_benchmarks():
    return {
        'cpu_integer_throughput_s': _time_it(_integer_ops),
        'cpu_float_throughput_s':   _time_it(_float_ops),
        'cpu_multithread_s':        _time_it(_multithread_ops),
    }

In [ ]:
%%writefile scripts/bench_memory.py
import time, numpy as np

N_REPEAT = 3

def _time_it(func, *args):
    times = []
    for _ in range(N_REPEAT):
        t0 = time.perf_counter()
        func(*args)
        times.append(time.perf_counter() - t0)
    return min(times)

def _sequential_read():
    arr = np.ones(32_000_000, dtype=np.float64)
    total = 0.0
    for _ in range(4):
        total += arr.sum()
    return total

def _random_access():
    size = 8_000_000
    arr  = np.arange(size, dtype=np.float64)
    idx  = np.random.randint(0, size, size=500_000)
    return arr[idx].sum()

def _cache_test(size_bytes):
    n = size_bytes // 8
    arr = np.ones(n, dtype=np.float64)
    n_ops = max(1, 64_000_000 // n)
    t0 = time.perf_counter()
    for _ in range(n_ops):
        arr += 1.0
    return time.perf_counter() - t0

def _write_bandwidth():
    arr = np.empty(32_000_000, dtype=np.float64)
    for _ in range(4):
        arr[:] = 3.14
    return arr.sum()

def run_memory_benchmarks():
    return {
        'mem_sequential_read_s':  _time_it(_sequential_read),
        'mem_random_access_s':    _time_it(_random_access),
        'mem_write_bandwidth_s':  _time_it(_write_bandwidth),
        'cache_l1_s':             _cache_test(32   * 1024),
        'cache_l2_s':             _cache_test(256  * 1024),
        'cache_l3_s':             _cache_test(8    * 1024 * 1024),
        'cache_ram_s':            _cache_test(64   * 1024 * 1024),
    }

In [ ]:
%%writefile scripts/bench_workloads.py
import time, sqlite3, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

N_REPEAT = 3
RANDOM_SEED = 42

def _time_it(func, *args):
    times = []
    for _ in range(N_REPEAT):
        t0 = time.perf_counter()
        func(*args)
        times.append(time.perf_counter() - t0)
    return min(times)

def _sort_numpy():
    rng = np.random.default_rng(RANDOM_SEED)
    arr = rng.random(5_000_000)
    np.sort(arr)

def _sort_python():
    rng = np.random.default_rng(RANDOM_SEED)
    lst = rng.integers(0, 10_000_000, size=1_000_000).tolist()
    sorted(lst)

def _join_pandas():
    rng = np.random.default_rng(RANDOM_SEED)
    n = 500_000
    orders = pd.DataFrame({'order_id': np.arange(n),
                           'customer_id': rng.integers(0, n//2, size=n),
                           'amount': rng.random(n)*1000})
    customers = pd.DataFrame({'customer_id': np.arange(n//2),
                               'age': rng.integers(18,80,size=n//2),
                               'score': rng.random(n//2)})
    return len(orders.merge(customers, on='customer_id', how='inner'))

def _ml_train():
    X, y = make_classification(n_samples=50_000, n_features=20,
                                n_informative=15, random_state=RANDOM_SEED)
    model = RandomForestClassifier(n_estimators=50, max_depth=10,
                                    n_jobs=-1, random_state=RANDOM_SEED)
    model.fit(X, y)

def _ml_inference():
    X, y = make_classification(n_samples=60_000, n_features=20,
                                n_informative=15, random_state=RANDOM_SEED)
    X_train, X_test = X[:50_000], X[50_000:]
    model = RandomForestClassifier(n_estimators=50, max_depth=10,
                                    n_jobs=-1, random_state=RANDOM_SEED)
    model.fit(X_train, y[:50_000])
    t0 = time.perf_counter()
    model.predict(X_test)
    return time.perf_counter() - t0

def _sql_benchmark():
    rng = np.random.default_rng(RANDOM_SEED)
    n = 100_000
    conn = sqlite3.connect(':memory:')
    cur  = conn.cursor()
    cur.execute('CREATE TABLE sales (id INT, region TEXT, product TEXT, amount REAL, quantity INT)')
    regions  = ['Nord','Sud','Est','Vest','Centru']
    products = ['Laptop','Telefon','Tableta','Monitor','Tastatura']
    rows = [(i, regions[rng.integers(0,5)], products[rng.integers(0,5)],
             float(rng.random()*5000), int(rng.integers(1,100))) for i in range(n)]
    cur.executemany('INSERT INTO sales VALUES (?,?,?,?,?)', rows)
    conn.commit()
    cur.execute('SELECT * FROM sales WHERE amount > 2500').fetchall()
    cur.execute('SELECT region, COUNT(*), AVG(amount) FROM sales GROUP BY region').fetchall()
    conn.close()

def run_workload_benchmarks():
    results = {}
    results['sort_numpy_s']    = _time_it(_sort_numpy)
    results['sort_python_s']   = _time_it(_sort_python)
    results['join_pandas_s']   = _time_it(_join_pandas)
    t0 = time.perf_counter(); _ml_train()
    results['ml_train_s']      = time.perf_counter() - t0
    results['ml_inference_s']  = _ml_inference()
    results['sql_benchmark_s'] = _time_it(_sql_benchmark)
    return results

In [ ]:
import json
from datetime import datetime
from scripts.system_info   import get_system_info
from scripts.bench_cpu     import run_cpu_benchmarks
from scripts.bench_memory  import run_memory_benchmarks
from scripts.bench_workloads import run_workload_benchmarks

sys_info = get_system_info()
for k, v in sys_info.items():
    print(f'   {k}: {v}')

cpu = run_cpu_benchmarks()
for k, v in cpu.items(): print(f'   {k}: {v:.4f}s')

mem = run_memory_benchmarks()
for k, v in mem.items(): print(f'   {k}: {v:.4f}s')

wl = run_workload_benchmarks()
for k, v in wl.items(): print(f'   {k}: {v:.4f}s')

results = {
    'timestamp':   datetime.now().isoformat(),
    'system_info': sys_info,
    'cpu':         cpu,
    'memory':      mem,
    'workloads':   wl,
}

os.makedirs('results', exist_ok=True)
with open('results/results_GoogleColab.json', 'w') as f:
    json.dump(results, f, indent=2)

In [ ]:
from google.colab import files
files.download('results/results_GoogleColab.json')